# AfroViewz 04: The Uganda holdout test set

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JBWandera/Capstone/blob/afroviewz/notebooks/04_uganda_holdout.ipynb)

**Workflow stage:** test set construction.

Every training image comes from outside Uganda. To know how AfroViewz does on **our** roads, a separate test set is built from Ugandan Mapillary street photos (CC BY-SA 4.0, each photo credited to its contributor). These images are never used for training or for choosing settings; they are scored once, at the end.

Most street photos show no pothole, and labelling 3,000 photos by hand is not realistic in the time available. So:
1. the first trained model scores every candidate photo;
2. the 350 photos it is most confident about, plus 150 random others, are uploaded to Roboflow with the model's boxes as **pre-labels**;
3. a person checks every photo in Roboflow Annotate: removes wrong boxes, adds missed potholes, fixes edges. The labels count as ground truth only after this review;
4. the reviewed set is exported and scored in notebook 07.

The 150 random photos matter: they test false alarms on ordinary Ugandan streets, and they include potholes the model missed, which keeps the ranking step from flattering the model.

## Setup

In [ ]:
# Works on Colab and locally. On Colab: mounts Drive, gets the code, installs it.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/Capstone"):
        subprocess.run(["git", "clone", "-q", "-b", "afroviewz", "https://github.com/JBWandera/Capstone.git", "/content/Capstone"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/Capstone", "pull", "-q"], check=True)
    os.chdir("/content/Capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
    sys.path.insert(0, "/content/Capstone/src")
from afroviewz import paths
print("data root:", paths.data_root())


In [ ]:
def secret(name):
    """Colab Secrets on Colab, environment variables elsewhere. Never printed."""
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

ROBOFLOW_API_KEY = secret("ROBOFLOW_API_KEY")


In [ ]:
import json
import pandas as pd
from afroviewz.data import coco as C, labelling
from afroviewz.train import runner

UG = paths.interim_dir("uganda_mapillary")
ug = C.load(UG / "coco.json")
meta = pd.read_csv(UG / "metadata.csv")
print(len(ug["images"]), "candidate photos")
meta.groupby("area").size()


## 1. Score every candidate with the first trained model

In [ ]:
MODEL_RUN = "E2_rfdetr_small"  #@param {type:"string"}
model = runner.load_rfdetr(MODEL_RUN)
fn = runner.rfdetr_predict_fn(model, threshold=0.2)
from PIL import Image
scores, boxes = {}, {}
from tqdm.auto import tqdm
for i in tqdm(range(0, len(ug["images"]), 8)):
    chunk = ug["images"][i:i + 8]
    outs = fn([Image.open(UG / "images" / im["file_name"]).convert("RGB") for im in chunk])
    for im, (xyxy, sc) in zip(chunk, outs):
        scores[im["id"]] = sc.tolist()
        boxes[im["id"]] = [[x0, y0, x1 - x0, y1 - y0] for x0, y0, x1, y1 in xyxy.tolist()]
(UG / "prelabel_scores.json").write_text(json.dumps({"scores": scores, "boxes": boxes}))
pd.Series({k: max(v) if v else 0 for k, v in scores.items()}).describe()


## 2. Choose 500 photos and upload them with pre-labels

In [ ]:
chosen = labelling.rank_candidates(scores, ug, n_positive=350, n_negative=150)
project_id = labelling.upload(ROBOFLOW_API_KEY, "afroviewz-uganda-holdout", ug, UG / "images", chosen,
                              prelabels=boxes, batch_name="uganda-holdout-v1")
(UG / "holdout_selection.json").write_text(json.dumps(chosen))
print("uploaded", len(chosen), "photos to Roboflow project", project_id)


## 3. Review in Roboflow (by hand)

Open the project `afroviewz-uganda-holdout` in Roboflow, Annotate tab, batch `uganda-holdout-v1`. For every photo:
* delete boxes that are not potholes (patches, shadows, manholes, puddles on flat road);
* draw a box around every pothole that has none, including small and distant ones;
* tighten loose boxes.

Labelling guide: a pothole is a hole or depression in the road surface with broken edges. Cracks without a hole, repaired patches and drainage covers are not potholes. Gravel (murram) roads count when there is a clear hole.

When done, generate a version with **no preprocessing and no augmentation**.

## 4. Bring the reviewed labels back

In [ ]:
HOLDOUT_VERSION = 1  #@param {type:"integer"}
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
ver = rf.workspace().project("afroviewz-uganda-holdout").version(HOLDOUT_VERSION)
dst = paths.raw_dir("uganda_holdout_labels")
ver.download("coco", location=str(dst), overwrite=True)
from afroviewz.data import converters
hold, root = converters.from_roboflow_export(dst, "uganda_holdout")
rep = __import__("afroviewz.data.build", fromlist=["build"]).prepare_source(
    "uganda_holdout", hold, root, paths.interim_dir("uganda_holdout"), max_side=1600)
print(rep["after"])
